# ENGE707 Kaggle Competition: Model Evaluation

## Custom CNN and ResNet18 Comparison

This notebook evaluates the trained custom CNN baseline and the improved ResNet18 model on the same saved validation subset.

It supports **Task 5: Model Evaluation and Kaggle Submission** by producing validation metrics and evidence for selecting the model that will later generate the Kaggle submission in Notebook 6.

The notebook will:

1. Load the unchanged validation split created in Notebook 2.
2. Recreate the deterministic validation preprocessing pipeline.
3. Load the best custom CNN and ResNet18 checkpoints.
4. Evaluate both models on all validation images.
5. Compare accuracy, loss, precision, recall, F1-score, parameter count, and evaluation time.
6. Create confusion matrices and class-level comparisons.
7. Analyse representative misclassifications.
8. Select the final model using validation evidence only.
9. Save tables, figures, and a machine-readable model-selection record.

**No training occurs in this notebook.** Model weights are loaded for evaluation and must remain unchanged. Test images are not used for model selection.

## 1. Import the Required Libraries

The imports support file handling, image preprocessing, PyTorch evaluation, classification metrics, tables, and figures. No optimiser or training scheduler is required because this notebook does not update model weights.

In [ ]:
from pathlib import Path
import json
import random
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from torchvision.transforms import InterpolationMode

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
)

## 2. Set Reproducibility and the Evaluation Safety Switch

The same random seed of 42 is reused. The evaluation switch remains False until Notebook 4 has produced the ResNet18 checkpoint and training evidence.

Changing RUN_EVALUATION to True does not train either model. It only loads saved weights and evaluates them.

In [ ]:
RANDOM_SEED = 42
RUN_EVALUATION = False

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

print("Random seed:", RANDOM_SEED)
print("Run evaluation:", RUN_EVALUATION)

## 3. Locate Local or Kaggle Files

Locally, the notebook finds the project root from the saved split files. On Kaggle, it searches attached read-only inputs for the split, preprocessing configuration, training images, checkpoints, and saved training summaries.

Evaluation outputs are written only to outputs/evaluation and outputs/figures locally, or to /kaggle/working on Kaggle. Original images and labels are never modified.

In [ ]:
RUNNING_ON_KAGGLE = Path("/kaggle/input").exists()


def find_input_file(filename, required=True):
    """Find one named file among attached Kaggle inputs."""
    matches = sorted(Path("/kaggle/input").rglob(filename))

    if matches:
        return matches[0]

    if required:
        raise FileNotFoundError(
            f"Required Kaggle input was not found: {filename}"
        )

    return None


if RUNNING_ON_KAGGLE:
    VALIDATION_SPLIT_PATH = find_input_file("validation_split.csv")
    PREPROCESSING_CONFIG_PATH = find_input_file(
        "preprocessing_config.json"
    )
    CUSTOM_CNN_CHECKPOINT_PATH = find_input_file(
        "custom_cnn_best.pth",
        required=False,
    )
    RESNET18_CHECKPOINT_PATH = find_input_file(
        "resnet18_best.pth",
        required=False,
    )
    CUSTOM_CNN_HISTORY_PATH = find_input_file(
        "custom_cnn_training_history.csv",
        required=False,
    )
    RESNET18_HISTORY_PATH = find_input_file(
        "resnet18_training_history.csv",
        required=False,
    )
    CUSTOM_CNN_SUMMARY_PATH = find_input_file(
        "custom_cnn_training_summary.json",
        required=False,
    )
    RESNET18_SUMMARY_PATH = find_input_file(
        "resnet18_training_summary.json",
        required=False,
    )

    EVALUATION_OUTPUT_DIR = Path(
        "/kaggle/working/outputs/evaluation"
    )
    FIGURE_DIR = Path("/kaggle/working/outputs/figures")
else:
    possible_project_roots = [Path.cwd(), Path.cwd().parent]
    PROJECT_ROOT = next(
        (
            folder
            for folder in possible_project_roots
            if (folder / "splits" / "validation_split.csv").exists()
        ),
        None,
    )

    if PROJECT_ROOT is None:
        raise FileNotFoundError(
            "The project root could not be found."
        )

    VALIDATION_SPLIT_PATH = (
        PROJECT_ROOT / "splits" / "validation_split.csv"
    )
    PREPROCESSING_CONFIG_PATH = (
        PROJECT_ROOT
        / "outputs"
        / "preprocessing"
        / "preprocessing_config.json"
    )
    CUSTOM_CNN_CHECKPOINT_PATH = (
        PROJECT_ROOT / "models" / "custom_cnn_best.pth"
    )
    RESNET18_CHECKPOINT_PATH = (
        PROJECT_ROOT / "models" / "resnet18_best.pth"
    )
    CUSTOM_CNN_HISTORY_PATH = (
        PROJECT_ROOT
        / "outputs"
        / "models"
        / "custom_cnn_training_history.csv"
    )
    RESNET18_HISTORY_PATH = (
        PROJECT_ROOT
        / "outputs"
        / "models"
        / "resnet18_training_history.csv"
    )
    CUSTOM_CNN_SUMMARY_PATH = (
        PROJECT_ROOT
        / "outputs"
        / "models"
        / "custom_cnn_training_summary.json"
    )
    RESNET18_SUMMARY_PATH = (
        PROJECT_ROOT
        / "outputs"
        / "models"
        / "resnet18_training_summary.json"
    )

    EVALUATION_OUTPUT_DIR = (
        PROJECT_ROOT / "outputs" / "evaluation"
    )
    FIGURE_DIR = PROJECT_ROOT / "outputs" / "figures"

EVALUATION_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

print("Environment:", "Kaggle" if RUNNING_ON_KAGGLE else "Local")
print("Evaluation output folder:", EVALUATION_OUTPUT_DIR)
print("Figure folder:", FIGURE_DIR)

## 4. Load the Saved Validation Split and Configuration

The original labelled dataset is not split again. Both models use the exact same 12,732 validation images and the same class order, preventing an unfair comparison.

In [ ]:
with open(PREPROCESSING_CONFIG_PATH) as configuration_file:
    preprocessing_config = json.load(configuration_file)

validation_split = pd.read_csv(
    VALIDATION_SPLIT_PATH,
    dtype={
        "image_id": "string",
        "class": "string",
        "filename": "string",
        "group_id": "string",
    },
)

IMAGE_SIZE = int(preprocessing_config["image_size"])
BATCH_SIZE = int(preprocessing_config["batch_size"])
CLASS_NAMES = list(preprocessing_config["class_names"])
CLASS_TO_INDEX = dict(preprocessing_config["class_to_index"])
INDEX_TO_CLASS = {
    int(index): class_name
    for class_name, index in CLASS_TO_INDEX.items()
}
NUMBER_OF_CLASSES = len(CLASS_NAMES)

if RUNNING_ON_KAGGLE:
    probe_filename = validation_split.iloc[0]["filename"]
    image_matches = sorted(
        Path("/kaggle/input").rglob(probe_filename)
    )
    training_matches = [
        path
        for path in image_matches
        if "train" in {part.lower() for part in path.parent.parts}
    ]

    if not training_matches:
        raise FileNotFoundError(
            "The attached Kaggle inputs do not contain training images."
        )

    TRAIN_IMAGE_DIR = training_matches[0].parent
else:
    TRAIN_IMAGE_DIR = PROJECT_ROOT / "data" / "train" / "train"

assert TRAIN_IMAGE_DIR.exists()
assert len(validation_split) == 12732
assert set(validation_split["class"]) == set(CLASS_NAMES)
assert validation_split["label_index"].between(
    0,
    NUMBER_OF_CLASSES - 1,
).all()

print("Validation images:", f"{len(validation_split):,}")
print("Training-image folder:", TRAIN_IMAGE_DIR)
print("Image size:", f"{IMAGE_SIZE} x {IMAGE_SIZE}")
print("Classes:", CLASS_NAMES)

## 5. Check the Required Model Evidence

Notebook 5 requires the best checkpoint, history, and summary for both models. While RUN_EVALUATION is False, missing ResNet18 files are reported without stopping the notebook. When evaluation is enabled, every required file must exist.

In [ ]:
required_model_files = {
    "Custom CNN checkpoint": CUSTOM_CNN_CHECKPOINT_PATH,
    "ResNet18 checkpoint": RESNET18_CHECKPOINT_PATH,
    "Custom CNN history": CUSTOM_CNN_HISTORY_PATH,
    "ResNet18 history": RESNET18_HISTORY_PATH,
    "Custom CNN summary": CUSTOM_CNN_SUMMARY_PATH,
    "ResNet18 summary": RESNET18_SUMMARY_PATH,
}

model_file_status = pd.DataFrame(
    {
        "file": required_model_files.keys(),
        "path": [
            str(path) if path is not None else "Not attached"
            for path in required_model_files.values()
        ],
        "exists": [
            path is not None and Path(path).exists()
            for path in required_model_files.values()
        ],
    }
)

display(model_file_status)

if RUN_EVALUATION:
    missing_model_files = model_file_status.loc[
        ~model_file_status["exists"],
        "file",
    ].tolist()

    if missing_model_files:
        raise FileNotFoundError(
            "Evaluation cannot start. Missing: "
            + ", ".join(missing_model_files)
        )
else:
    print(
        "Evaluation remains disabled until both trained models exist."
    )

## 6. Select the Evaluation Device

CUDA is preferred on Kaggle, followed by Apple Metal locally, and then CPU. Evaluation uses inference mode and does not calculate gradients.

In [ ]:
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

USE_MIXED_PRECISION = DEVICE.type == "cuda"

print("Selected device:", DEVICE)
print("CUDA mixed precision:", USE_MIXED_PRECISION)
if DEVICE.type == "cuda":
    print("CUDA GPUs:", torch.cuda.device_count())

## 7. Recreate Deterministic Validation Preprocessing

Validation images receive RGB conversion, square padding, 224 x 224 resizing, tensor conversion, and ImageNet normalisation. No random flip, rotation, colour change, or other augmentation is applied.

In [ ]:
class SquarePad:
    """Add equal padding around the shorter sides of an image."""

    def __init__(self, fill=(0, 0, 0)):
        self.fill = fill

    def __call__(self, image):
        width, height = image.size
        largest_side = max(width, height)

        horizontal_padding = largest_side - width
        vertical_padding = largest_side - height

        left_padding = horizontal_padding // 2
        right_padding = horizontal_padding - left_padding
        top_padding = vertical_padding // 2
        bottom_padding = vertical_padding - top_padding

        return ImageOps.expand(
            image,
            border=(
                left_padding,
                top_padding,
                right_padding,
                bottom_padding,
            ),
            fill=self.fill,
        )


evaluation_transform = transforms.Compose(
    [
        SquarePad(),
        transforms.Resize(
            (IMAGE_SIZE, IMAGE_SIZE),
            interpolation=InterpolationMode.BILINEAR,
        ),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=preprocessing_config["normalisation_mean"],
            std=preprocessing_config["normalisation_std"],
        ),
    ]
)

print(evaluation_transform)

## 8. Create the Validation Dataset and Data Loader

The loader preserves validation order by using shuffle=False. This makes image-level predictions reproducible and allows every prediction to be linked back to its image ID.

In [ ]:
class FacialFeatureDataset(Dataset):
    """Load labelled facial-feature images for evaluation."""

    def __init__(self, dataframe, image_directory, transform):
        self.dataframe = dataframe.reset_index(drop=True).copy()
        self.image_directory = Path(image_directory)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]
        image_path = self.image_directory / row["filename"]

        with Image.open(image_path) as image:
            image = image.convert("RGB")
            image = self.transform(image)

        return {
            "image": image,
            "image_id": row["image_id"],
            "filename": row["filename"],
            "label": int(row["label_index"]),
            "class_name": row["class"],
        }


validation_dataset = FacialFeatureDataset(
    dataframe=validation_split,
    image_directory=TRAIN_IMAGE_DIR,
    transform=evaluation_transform,
)

NUM_WORKERS = (
    4
    if RUNNING_ON_KAGGLE
    else int(preprocessing_config["num_workers"])
)
PIN_MEMORY = DEVICE.type == "cuda"
loader_options = {}

if NUM_WORKERS > 0:
    loader_options = {
        "persistent_workers": True,
        "prefetch_factor": 2,
    }

validation_loader = DataLoader(
    validation_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    **loader_options,
)

print("Validation batches:", len(validation_loader))
print("Batch size:", BATCH_SIZE)
print("Shuffle:", False)

## 9. Recreate the Two Model Architectures

The custom CNN definition exactly matches Notebook 3. ResNet18 is created without downloading weights because its learned weights will come from the saved Notebook 4 checkpoint.

In [ ]:
class ConvolutionBlock(nn.Module):
    """Extract and refine features before reducing image size."""

    def __init__(self, input_channels, output_channels):
        super().__init__()

        self.layers = nn.Sequential(
            nn.Conv2d(
                input_channels,
                output_channels,
                kernel_size=3,
                padding=1,
                bias=False,
            ),
            nn.BatchNorm2d(output_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(
                output_channels,
                output_channels,
                kernel_size=3,
                padding=1,
                bias=False,
            ),
            nn.BatchNorm2d(output_channels),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )

    def forward(self, images):
        return self.layers(images)


class CustomCNN(nn.Module):
    """Custom CNN baseline from Notebook 3."""

    def __init__(self, number_of_classes=8, dropout_rate=0.40):
        super().__init__()

        self.features = nn.Sequential(
            ConvolutionBlock(3, 32),
            ConvolutionBlock(32, 64),
            ConvolutionBlock(64, 128),
            ConvolutionBlock(128, 256),
        )
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(p=dropout_rate),
            nn.Linear(256, number_of_classes),
        )

    def forward(self, images):
        features = self.features(images)
        pooled_features = self.global_pool(features)
        return self.classifier(pooled_features)


def create_resnet18(number_of_classes=8):
    """Create the ResNet18 architecture expected from Notebook 4."""
    model = models.resnet18(weights=None)
    input_features = model.fc.in_features
    model.fc = nn.Linear(input_features, number_of_classes)
    return model


model_builders = {
    "Custom CNN": lambda: CustomCNN(
        number_of_classes=NUMBER_OF_CLASSES,
        dropout_rate=0.40,
    ),
    "ResNet18": lambda: create_resnet18(
        number_of_classes=NUMBER_OF_CLASSES,
    ),
}

checkpoint_paths = {
    "Custom CNN": CUSTOM_CNN_CHECKPOINT_PATH,
    "ResNet18": RESNET18_CHECKPOINT_PATH,
}

print("Model architectures defined.")
print("Expected output classes:", NUMBER_OF_CLASSES)

## 10. Load and Verify the Best Checkpoints

DataParallel prefixes are removed when necessary. Checkpoint class names must match the saved preprocessing configuration exactly.

In [ ]:
def clean_state_dict(state_dict):
    """Remove a DataParallel module prefix when present."""
    return {
        key.removeprefix("module."): value
        for key, value in state_dict.items()
    }


def load_best_model(model_name, device):
    """Load and verify one best-validation checkpoint."""
    checkpoint_path = Path(checkpoint_paths[model_name])
    checkpoint = torch.load(
        checkpoint_path,
        map_location=device,
        weights_only=False,
    )

    checkpoint_classes = checkpoint.get("class_names")

    if checkpoint_classes is not None:
        assert list(checkpoint_classes) == CLASS_NAMES

    model = model_builders[model_name]().to(device)
    model.load_state_dict(
        clean_state_dict(checkpoint["model_state_dict"])
    )
    model.eval()

    example = torch.zeros(
        1,
        3,
        IMAGE_SIZE,
        IMAGE_SIZE,
        device=device,
    )

    with torch.inference_mode():
        example_logits = model(example)

    assert tuple(example_logits.shape) == (
        1,
        NUMBER_OF_CLASSES,
    )

    return model, checkpoint


if RUN_EVALUATION:
    loaded_models = {}
    loaded_checkpoints = {}

    for model_name in model_builders:
        loaded_models[model_name], loaded_checkpoints[model_name] = (
            load_best_model(model_name, DEVICE)
        )
        print(model_name, "checkpoint loaded and verified.")
else:
    loaded_models = {}
    loaded_checkpoints = {}
    print("Checkpoint loading skipped.")

## 11. Define Reusable Validation Evaluation

The function calculates loss and accuracy across all validation batches. It also stores each image ID, true label, predicted label, and prediction confidence for class-level and error analysis.

In [ ]:
def evaluate_classifier(
    model,
    data_loader,
    class_names,
    device,
    use_mixed_precision,
):
    """Evaluate one model without changing its weights."""
    model.eval()
    loss_function = nn.CrossEntropyLoss(reduction="sum")

    total_loss = 0.0
    correct_predictions = 0
    evaluated_images = 0
    prediction_rows = []

    evaluation_start = time.perf_counter()

    with torch.inference_mode():
        for batch in data_loader:
            images = batch["image"].to(
                device,
                non_blocking=device.type == "cuda",
            )
            labels = batch["label"].to(
                device,
                non_blocking=device.type == "cuda",
            )

            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16,
                enabled=use_mixed_precision,
            ):
                logits = model(images)
                loss = loss_function(logits, labels)

            probabilities = torch.softmax(
                logits.float(),
                dim=1,
            )
            confidences, predicted_indices = probabilities.max(dim=1)

            batch_size = labels.size(0)
            total_loss += loss.item()
            correct_predictions += (
                predicted_indices == labels
            ).sum().item()
            evaluated_images += batch_size

            for position in range(batch_size):
                true_index = int(labels[position].item())
                predicted_index = int(
                    predicted_indices[position].item()
                )

                prediction_rows.append(
                    {
                        "image_id": batch["image_id"][position],
                        "filename": batch["filename"][position],
                        "true_index": true_index,
                        "predicted_index": predicted_index,
                        "true_class": class_names[true_index],
                        "predicted_class": class_names[
                            predicted_index
                        ],
                        "confidence": float(
                            confidences[position].item()
                        ),
                        "correct": true_index == predicted_index,
                    }
                )

    evaluation_seconds = time.perf_counter() - evaluation_start

    metrics = {
        "validation_loss": total_loss / evaluated_images,
        "validation_accuracy": (
            correct_predictions / evaluated_images
        ),
        "evaluated_images": evaluated_images,
        "evaluation_seconds": evaluation_seconds,
    }

    return metrics, pd.DataFrame(prediction_rows)

## 12. Evaluate Both Models on the Same Validation Images

Before evaluation, every model parameter is copied. After evaluation, the copies are compared with the current parameters to prove that no weights were updated.

In [ ]:
evaluation_results = {}
prediction_tables = {}
weights_unchanged = {}

if RUN_EVALUATION:
    for model_name, model in loaded_models.items():
        parameter_snapshot = {
            name: parameter.detach().cpu().clone()
            for name, parameter in model.named_parameters()
        }

        metrics, predictions = evaluate_classifier(
            model=model,
            data_loader=validation_loader,
            class_names=CLASS_NAMES,
            device=DEVICE,
            use_mixed_precision=USE_MIXED_PRECISION,
        )

        weights_unchanged[model_name] = all(
            torch.equal(
                parameter.detach().cpu(),
                parameter_snapshot[name],
            )
            for name, parameter in model.named_parameters()
        )

        evaluation_results[model_name] = metrics
        prediction_tables[model_name] = predictions

        predictions.to_csv(
            EVALUATION_OUTPUT_DIR
            / (
                model_name.lower().replace(" ", "_")
                + "_validation_predictions.csv"
            ),
            index=False,
        )

        print(
            f"{model_name}: "
            f"accuracy {metrics['validation_accuracy']:.4f}, "
            f"loss {metrics['validation_loss']:.4f}, "
            f"time {metrics['evaluation_seconds']:.1f} seconds"
        )
else:
    print("Validation evaluation skipped.")

## 13. Calculate Overall and Class-Level Metrics

Accuracy is the competition metric. Precision, recall, F1-score, and support provide additional evidence about performance across all eight classes.

In [ ]:
classification_reports = {}
overall_rows = []

if RUN_EVALUATION:
    for model_name, predictions in prediction_tables.items():
        report = classification_report(
            predictions["true_index"],
            predictions["predicted_index"],
            labels=list(range(NUMBER_OF_CLASSES)),
            target_names=CLASS_NAMES,
            output_dict=True,
            zero_division=0,
        )

        report_table = (
            pd.DataFrame(report)
            .transpose()
            .reset_index()
            .rename(columns={"index": "class_or_average"})
        )
        classification_reports[model_name] = report_table

        report_table.to_csv(
            EVALUATION_OUTPUT_DIR
            / (
                model_name.lower().replace(" ", "_")
                + "_classification_report.csv"
            ),
            index=False,
        )

        model = loaded_models[model_name]
        parameter_count = sum(
            parameter.numel()
            for parameter in model.parameters()
        )

        overall_rows.append(
            {
                "model": model_name,
                "validation_accuracy": evaluation_results[
                    model_name
                ]["validation_accuracy"],
                "validation_loss": evaluation_results[
                    model_name
                ]["validation_loss"],
                "macro_precision": report["macro avg"][
                    "precision"
                ],
                "macro_recall": report["macro avg"]["recall"],
                "macro_f1": report["macro avg"]["f1-score"],
                "weighted_f1": report["weighted avg"][
                    "f1-score"
                ],
                "parameter_count": parameter_count,
                "evaluation_seconds": evaluation_results[
                    model_name
                ]["evaluation_seconds"],
            }
        )

    model_comparison = pd.DataFrame(overall_rows)
    model_comparison.to_csv(
        EVALUATION_OUTPUT_DIR / "model_comparison.csv",
        index=False,
    )

    display(model_comparison.round(4))
else:
    model_comparison = None
    print("Metric tables are pending evaluation.")

## 14. Plot the Overall Model Comparison

The comparison figure presents validation accuracy and macro F1-score on the same percentage scale. Both measures are calculated from the same validation predictions.

In [ ]:
MODEL_COMPARISON_FIGURE_PATH = (
    FIGURE_DIR / "figure_15_model_comparison.png"
)

if model_comparison is not None:
    comparison_plot = model_comparison.set_index("model")[
        ["validation_accuracy", "macro_f1"]
    ] * 100
    comparison_plot.columns = [
        "Validation accuracy",
        "Macro F1-score",
    ]

    axis = comparison_plot.plot(
        kind="bar",
        figsize=(9, 5),
        color=["#3B82F6", "#F59E0B"],
        edgecolor="black",
    )
    axis.set_title("Validation Performance by Model")
    axis.set_xlabel("Model")
    axis.set_ylabel("Score (%)")
    axis.set_ylim(0, 100)
    axis.tick_params(axis="x", rotation=0)
    axis.grid(axis="y", alpha=0.3)
    axis.legend(loc="lower right")

    for container in axis.containers:
        axis.bar_label(container, fmt="%.2f", padding=3)

    plt.tight_layout()
    plt.savefig(
        MODEL_COMPARISON_FIGURE_PATH,
        dpi=300,
        bbox_inches="tight",
    )
    plt.show()

    print("Saved:", MODEL_COMPARISON_FIGURE_PATH)
else:
    print("Model-comparison figure is pending evaluation.")

## 15. Create Confusion Matrices

A confusion matrix shows how often each true class is predicted as every possible class. Separate figures are saved for the custom CNN and ResNet18.

In [ ]:
confusion_matrices = {}
confusion_figure_paths = {
    "Custom CNN": (
        FIGURE_DIR / "figure_16_custom_cnn_confusion_matrix.png"
    ),
    "ResNet18": (
        FIGURE_DIR / "figure_17_resnet18_confusion_matrix.png"
    ),
}

if RUN_EVALUATION:
    for model_name, predictions in prediction_tables.items():
        matrix = confusion_matrix(
            predictions["true_index"],
            predictions["predicted_index"],
            labels=list(range(NUMBER_OF_CLASSES)),
        )
        confusion_matrices[model_name] = matrix

        matrix_table = pd.DataFrame(
            matrix,
            index=CLASS_NAMES,
            columns=CLASS_NAMES,
        )
        matrix_table.to_csv(
            EVALUATION_OUTPUT_DIR
            / (
                model_name.lower().replace(" ", "_")
                + "_confusion_matrix.csv"
            )
        )

        figure, axis = plt.subplots(figsize=(9, 8))
        image = axis.imshow(matrix, cmap="Blues")
        figure.colorbar(image, ax=axis)

        axis.set(
            xticks=np.arange(NUMBER_OF_CLASSES),
            yticks=np.arange(NUMBER_OF_CLASSES),
            xticklabels=CLASS_NAMES,
            yticklabels=CLASS_NAMES,
            xlabel="Predicted class",
            ylabel="True class",
            title=f"{model_name} Confusion Matrix",
        )
        plt.setp(
            axis.get_xticklabels(),
            rotation=45,
            ha="right",
            rotation_mode="anchor",
        )

        threshold = matrix.max() / 2

        for row in range(NUMBER_OF_CLASSES):
            for column in range(NUMBER_OF_CLASSES):
                axis.text(
                    column,
                    row,
                    f"{matrix[row, column]:,}",
                    ha="center",
                    va="center",
                    color=(
                        "white"
                        if matrix[row, column] > threshold
                        else "black"
                    ),
                    fontsize=8,
                )

        plt.tight_layout()
        plt.savefig(
            confusion_figure_paths[model_name],
            dpi=300,
            bbox_inches="tight",
        )
        plt.show()

        print("Saved:", confusion_figure_paths[model_name])
else:
    print("Confusion matrices are pending evaluation.")

## 16. Compare Per-Class Accuracy

Class-level accuracy is the proportion of images from one true class that were predicted correctly. This identifies difficult feature categories that overall accuracy could hide.

In [ ]:
PER_CLASS_FIGURE_PATH = (
    FIGURE_DIR / "figure_18_per_class_performance.png"
)

per_class_rows = []

if RUN_EVALUATION:
    for model_name, matrix in confusion_matrices.items():
        class_totals = matrix.sum(axis=1)
        class_correct = np.diag(matrix)
        class_accuracy = np.divide(
            class_correct,
            class_totals,
            out=np.zeros_like(
                class_correct,
                dtype=float,
            ),
            where=class_totals != 0,
        )

        for class_name, accuracy, support in zip(
            CLASS_NAMES,
            class_accuracy,
            class_totals,
        ):
            per_class_rows.append(
                {
                    "model": model_name,
                    "class": class_name,
                    "class_accuracy": accuracy,
                    "support": int(support),
                }
            )

    per_class_performance = pd.DataFrame(per_class_rows)
    per_class_performance.to_csv(
        EVALUATION_OUTPUT_DIR / "per_class_performance.csv",
        index=False,
    )

    per_class_plot = (
        per_class_performance.pivot(
            index="class",
            columns="model",
            values="class_accuracy",
        )
        .reindex(CLASS_NAMES)
        * 100
    )

    axis = per_class_plot.plot(
        kind="bar",
        figsize=(12, 6),
        color=["#3B82F6", "#F59E0B"],
        edgecolor="black",
    )
    axis.set_title("Validation Accuracy by Class")
    axis.set_xlabel("Class")
    axis.set_ylabel("Accuracy (%)")
    axis.set_ylim(0, 100)
    axis.tick_params(axis="x", rotation=35)
    axis.grid(axis="y", alpha=0.3)

    plt.tight_layout()
    plt.savefig(
        PER_CLASS_FIGURE_PATH,
        dpi=300,
        bbox_inches="tight",
    )
    plt.show()

    display(per_class_performance.round(4))
else:
    per_class_performance = None
    print("Per-class comparison is pending evaluation.")

## 17. Analyse Common Misclassifications

Incorrect predictions are grouped by true and predicted class. This reveals the class pairs that each model confuses most often.

In [ ]:
misclassification_tables = {}
confusion_pair_tables = {}

if RUN_EVALUATION:
    for model_name, predictions in prediction_tables.items():
        mistakes = predictions.loc[
            ~predictions["correct"]
        ].copy()
        mistakes = mistakes.sort_values(
            "confidence",
            ascending=False,
        )
        misclassification_tables[model_name] = mistakes

        confusion_pairs = (
            mistakes.groupby(
                ["true_class", "predicted_class"],
                observed=True,
            )
            .size()
            .reset_index(name="count")
            .sort_values("count", ascending=False)
        )
        confusion_pair_tables[model_name] = confusion_pairs

        file_stem = model_name.lower().replace(" ", "_")
        mistakes.to_csv(
            EVALUATION_OUTPUT_DIR
            / f"{file_stem}_misclassified_images.csv",
            index=False,
        )
        confusion_pairs.to_csv(
            EVALUATION_OUTPUT_DIR
            / f"{file_stem}_confusion_pairs.csv",
            index=False,
        )

        print(
            model_name,
            "misclassified images:",
            f"{len(mistakes):,}",
        )
        display(confusion_pairs.head(10))
else:
    print("Misclassification tables are pending evaluation.")

## 18. Select the Final Model Using Validation Evidence

The final model is selected before viewing Kaggle test labels. Models are ranked by validation accuracy, then macro F1-score, and finally lower validation loss. This avoids choosing a model using hidden-test feedback.

In [ ]:
SELECTED_MODEL_PATH = (
    EVALUATION_OUTPUT_DIR / "selected_model.json"
)

if model_comparison is not None:
    ranked_models = model_comparison.sort_values(
        by=[
            "validation_accuracy",
            "macro_f1",
            "validation_loss",
        ],
        ascending=[False, False, True],
    ).reset_index(drop=True)

    selected_row = ranked_models.iloc[0]
    selected_model_name = selected_row["model"]

    selected_model_record = {
        "selected_model": selected_model_name,
        "selection_basis": (
            "Highest validation accuracy, followed by macro F1 "
            "and lower validation loss"
        ),
        "validation_accuracy": float(
            selected_row["validation_accuracy"]
        ),
        "validation_loss": float(
            selected_row["validation_loss"]
        ),
        "macro_f1": float(selected_row["macro_f1"]),
        "validation_images": int(len(validation_dataset)),
        "test_images_used_for_selection": False,
    }

    with open(SELECTED_MODEL_PATH, "w") as selected_file:
        json.dump(
            selected_model_record,
            selected_file,
            indent=4,
        )

    display(ranked_models.round(4))
    print("Selected model:", selected_model_name)
    print("Saved:", SELECTED_MODEL_PATH)
else:
    selected_model_name = None
    selected_model_record = None
    print("Final model selection is pending evaluation.")

## 19. Display Representative High-Confidence Errors

The selected model’s most confident incorrect predictions are displayed using the original validation images. These examples support discussion of unclear class boundaries and difficult visual cases.

In [ ]:
MISCLASSIFICATION_FIGURE_PATH = (
    FIGURE_DIR / "figure_19_misclassified_examples.png"
)

if selected_model_name is not None:
    selected_errors = (
        misclassification_tables[selected_model_name]
        .head(12)
        .reset_index(drop=True)
    )

    figure, axes = plt.subplots(3, 4, figsize=(14, 10))

    for axis, (_, row) in zip(
        axes.flat,
        selected_errors.iterrows(),
    ):
        image_path = TRAIN_IMAGE_DIR / row["filename"]

        with Image.open(image_path) as image:
            axis.imshow(image.convert("RGB"))

        axis.set_title(
            f"True: {row['true_class']}\n"
            f"Predicted: {row['predicted_class']} "
            f"({row['confidence']:.1%})",
            fontsize=9,
        )
        axis.axis("off")

    for axis in axes.flat[len(selected_errors):]:
        axis.axis("off")

    figure.suptitle(
        f"High-Confidence Errors: {selected_model_name}",
        fontsize=14,
    )
    plt.tight_layout()
    plt.savefig(
        MISCLASSIFICATION_FIGURE_PATH,
        dpi=300,
        bbox_inches="tight",
    )
    plt.show()

    print("Saved:", MISCLASSIFICATION_FIGURE_PATH)
else:
    print("Misclassification examples are pending evaluation.")

## 20. Save the Evaluation Summary and Run Final Checks

The final checks verify complete and comparable evaluation, valid class labels, unchanged weights, saved evidence, and the absence of test-data use during model selection.

In [ ]:
EVALUATION_SUMMARY_PATH = (
    EVALUATION_OUTPUT_DIR / "evaluation_summary.json"
)

if RUN_EVALUATION:
    summary_records = model_comparison.to_dict(
        orient="records"
    )

    evaluation_summary = {
        "validation_images": int(len(validation_dataset)),
        "class_names": CLASS_NAMES,
        "models": summary_records,
        "selected_model": selected_model_record,
        "validation_transform_is_random": False,
        "test_images_used": False,
    }

    with open(EVALUATION_SUMMARY_PATH, "w") as summary_file:
        json.dump(
            evaluation_summary,
            summary_file,
            indent=4,
        )

    expected_validation_ids = (
        validation_split["image_id"].astype("string").tolist()
    )

    prediction_counts_are_complete = all(
        len(table) == len(validation_dataset)
        for table in prediction_tables.values()
    )
    prediction_order_is_correct = all(
        table["image_id"].astype("string").tolist()
        == expected_validation_ids
        for table in prediction_tables.values()
    )
    predicted_classes_are_valid = all(
        set(table["predicted_class"]).issubset(
            set(CLASS_NAMES)
        )
        for table in prediction_tables.values()
    )

    expected_output_paths = [
        EVALUATION_OUTPUT_DIR / "model_comparison.csv",
        EVALUATION_OUTPUT_DIR
        / "custom_cnn_classification_report.csv",
        EVALUATION_OUTPUT_DIR
        / "resnet18_classification_report.csv",
        EVALUATION_OUTPUT_DIR
        / "custom_cnn_confusion_matrix.csv",
        EVALUATION_OUTPUT_DIR
        / "resnet18_confusion_matrix.csv",
        EVALUATION_OUTPUT_DIR / "per_class_performance.csv",
        EVALUATION_SUMMARY_PATH,
        SELECTED_MODEL_PATH,
        MODEL_COMPARISON_FIGURE_PATH,
        confusion_figure_paths["Custom CNN"],
        confusion_figure_paths["ResNet18"],
        PER_CLASS_FIGURE_PATH,
        MISCLASSIFICATION_FIGURE_PATH,
    ]

    evaluation_checks = pd.DataFrame(
        {
            "check": [
                "Both trained models were evaluated",
                "Each model evaluated all validation images",
                "Validation prediction order was preserved",
                "All predicted classes are valid",
                "Evaluation left every model weight unchanged",
                "Both models used the same validation loader",
                "No random validation augmentation was used",
                "No test images were used for model selection",
                "All required evidence files exist",
            ],
            "result": [
                set(evaluation_results)
                == {"Custom CNN", "ResNet18"},
                prediction_counts_are_complete,
                prediction_order_is_correct,
                predicted_classes_are_valid,
                all(weights_unchanged.values()),
                True,
                True,
                True,
                all(path.exists() for path in expected_output_paths),
            ],
            "expected": [True] * 9,
        }
    )

    display(evaluation_checks)
    assert evaluation_checks["result"].all()

    TASK_5_EVALUATION_STATUS = "Completed and verified"
    print("Task 5 evaluation status:", TASK_5_EVALUATION_STATUS)
    print("Kaggle submission remains for Notebook 6.")
else:
    evaluation_summary = None
    TASK_5_EVALUATION_STATUS = (
        "Pending - waiting for the ResNet18 training evidence"
    )
    print("Task 5 evaluation status:", TASK_5_EVALUATION_STATUS)

# Notebook 5 Conclusion - Pending Model Evaluation

This notebook is ready to evaluate the custom CNN and ResNet18 on the same 12,732 validation images. Evaluation is currently disabled because the improved ResNet18 checkpoint and training evidence must first be created in Notebook 4.

After both models are available, set RUN_EVALUATION to True and run the notebook. Replace this conclusion with the actual comparison, strongest and weakest classes, common misclassifications, selected final model, and reasons for that selection.

Kaggle test prediction and submission will be completed separately in Notebook 6. Test images are not used in this evaluation notebook.